# MPNN and RFD3 from a pdb

In [1]:
import numpy as np
import pandas as pd

from biotite.structure.io import load_structure
from mpnn.inference_engines.mpnn import MPNNInferenceEngine
from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3.utils.inference import InferenceInput

from biotite.structure import get_residue_starts
from biotite.sequence import ProteinSequence
from biotite.structure import rmsd, superimpose

from atomworks.constants import PROTEIN_BACKBONE_ATOM_NAMES
from atomworks.io.utils.io_utils import to_cif_file

Environment variable CCD_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
Environment variable PDB_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
07:46:13 INFO rdkit: Enabling RDKit 2025.03.6 jupyter extensions


In [2]:
path = "./volume_data/job_0_0.pdb"
x = load_structure(path)
atom_array = x

/paperspace/apps/miniconda3/envs/foundry/lib/python3.12/site-packages/biotite/structure/io/pdb/file.py:470: UserWarning: 1596 elements were guessed from atom name
  warnings.warn(


In [6]:
engine_config = {
    'model_type': "protein_mpnn",
    "is_legacy_weights": True,
    "out_directory": "new",
    "write_structures": True,
    "write_fasta": False,
}

input_configs = [
    {
        "batch_size": 8,
        "remove_waters": True,
        "fixed_chains": ["B"],
        "temperature": 0.1,
        "name": "tests",
    }
]

model = MPNNInferenceEngine(**engine_config)
mpnn_outputs = model.run(input_dicts=input_configs, atom_arrays=[atom_array])

07:54:25 INFO mpnn.inference_engines.mpnn: [rank: 0] Loading legacy MPNN weights.


07:54:27 INFO mpnn.utils.inference: Annotated AtomArray has 5438 atoms 
07:54:27 INFO mpnn.inference_engines.mpnn: [rank: 0] Running MPNN inference for input 0, batch 0...


In [4]:
print(f"Generated {len(mpnn_outputs)} designed sequences:\n")

for i, item in enumerate(mpnn_outputs):
    res_starts = get_residue_starts(item.atom_array)
    # Convert 3-letter codes to 1-letter using Biotite
    seq_1letter = ''.join(
        ProteinSequence.convert_letter_3to1(res_name)
        for res_name in item.atom_array.res_name[res_starts]
    )
    print(f"Sequence {i+1}: {seq_1letter}")

Generated 8 designed sequences:

Sequence 1: MPKRTQSELDETKKELLSLMKLTIKLSKDPSVDKEKLLKLYKEAEEKLAKLSKEITKLQLELAEAQQKGENAEALQKKIDEKLEELEKLVKETKETIIKESGDKSLLEGSDLQVCLPKGPTCCSRKMEEKYQLTARLNMEQLLQSASKELKFLIIQNAAVFQEAFEIVVRHAKNYTNAMFKNNYPSLTPQAFEFVGEFFTDVSLYILGSDINVDDMVNELFDSLFPVIYTQLMNPGLPDSALDINECLRGARRDLKVFGNFPKLIMTQVSKSLQVTRIFLQALNLGIEVINTTDHLKFSKDCGRMLTRMWYCSYCQGLMMVKPCGGYCNVVMQGCMAGVVEIDKYWREYILSLEELVNGMYRIYDMENVLLGLFSTIHDSIQYVQKNAGKLTTTIGKLC
Sequence 2: MKKITDKEIEKTIKELLSLMRLLRKKSKDPNVDKEKLLKKYEEAEKKLEELSKKYTELKLKKAKAEQEGTNTKEYEKKIIELYEELEKLVKELKEFLIKESGDKSLLEGSDLQVCLPKGPTCCSRKMEEKYQLTARLNMEQLLQSASKELKFLIIQNAAVFQEAFEIVVRHAKNYTNAMFKNNYPSLTPQAFEFVGEFFTDVSLYILGSDINVDDMVNELFDSLFPVIYTQLMNPGLPDSALDINECLRGARRDLKVFGNFPKLIMTQVSKSLQVTRIFLQALNLGIEVINTTDHLKFSKDCGRMLTRMWYCSYCQGLMMVKPCGGYCNVVMQGCMAGVVEIDKYWREYILSLEELVNGMYRIYDMENVLLGLFSTIHDSIQYVQKNAGKLTTTIGKLC
Sequence 3: APPISDAEIQQTIKELLSLIRLARKLAKDPSVDKAALRALAEEAEARLAELSKQYTALKLAAAKAEQAGTDTAAWEAQIRAIYAELQALVEEMRATLIALSGDQSLLEGSDLQVCLPKGPTCCSRKMEEKY

In [ ]:
inference_engine = RF3InferenceEngine(ckpt_path='rf3', verbose=False)
input_structure = InferenceInput.from_atom_array(mpnn_outputs[2].atom_array,example_id="binder", template_selection="B")
rf3_outputs = inference_engine.run(inputs=input_structure)

06:17:49 WARNING atomworks.io: The `extra_fields` argument will be ignored if there is no CIF file input.
06:17:49 WARNING atomworks.io: Adding missing atoms will erase extra fields. If you just want to load a structure with the given extra fields, you should probably use the much faster 'load_any' function from atomworks.io.utils.io_utils instead of 'parse'. Parse is meant for cleaning up structures from the RCSB PDB.


06:17:49 INFO rf3.inference_engines.rf3: [rank: 0] Loading checkpoint from /home/ubuntu/.foundry/checkpoints/rf3_foundry_01_24_latest_remapped.ckpt...
06:17:50 WARNING atomworks.ml: Using element type for atom names of atomized tokens.
Using bfloat16 Automatic Mixed Precision (AMP)
06:17:53 WARNING rf3.inference_engines.rf3: [rank: 0] out_dir is None - results will be returned in memory! If you want to save to disk, please provide an out_dir.
06:17:53 INFO rf3.inference_engines.rf3: [rank: 0] Found 1 structures to predict!
06:17:53 INFO rf3.inference_engines.rf3: [rank: 0] Predicting structure 1/1: binder
06:17:53 WARNING atomworks.ml: Cached data not found for ALA at /net/tukwila/lschaaf/datahub/MACE-OMOL-Jul2025/mace_embeddings/A/ALA/ALA.pt
06:17:53 WARNING atomworks.ml: Cached data not found for ARG at /net/tukwila/lschaaf/datahub/MACE-OMOL-Jul2025/mace_embeddings/A/ARG/ARG.pt
06:17:53 WARNING atomworks.ml: Cached data not found for ASN at /net/tukwila/lschaaf/datahub/MACE-OMOL-Jul2

In [ ]:
rf3_output = rf3_outputs['binder'][0]
res_starts = get_residue_starts(rf3_output.atom_array)
    # Convert 3-letter codes to 1-letter using Biotite
seq_1letter = ''.join(
    ProteinSequence.convert_letter_3to1(res_name)
    for res_name in rf3_output.atom_array.res_name[res_starts]
)
print(f"Sequence {i+1}: {seq_1letter}")

Sequence 8: GFLKKLVENGYITYEEAKEMGVSDETLEYLIENNYITSIEENGKTLYVITLEGIKYMKENNLGSDLQVCLPKGPTCCSRKMEEKYQLTARLNMEQLLQSASMELKFLIIQNAAVFQEAFEIVVRHAKNYTNAMFKNNYPSLTPQAFEFVGEFFTDVSLYILGSDINVDDMVNELFDSLFPVIYTQLMNPGLPDSALDINECLRGARRDLKVFGNFPKLIMTQVSKSLQVTRIFLQALNLGIEVINTTDHLKFSKDCGRMLTRMWYCSYCQGLMMVKPCGGYCNVVMQGCMAGVVEIDKYWREYILSLEELVNGMYRIYDMENVLLGLFSTIHDSIQYVQKNAGKLTTTIGKLCAHSQQRQYRSAYYPEDLFIDKKVLKVAHVEHEETLSSRRRELIQKLKSFISFYSALPGYICSHSPVAENDTLCWNGQELVERYSQKAARNGMKNQFNLHELKMKGPEPVVSQIIDKLKHINQLLRTMS


In [ ]:
rf3_output = rf3_outputs['binder'][0]
res_starts = get_residue_starts(rf3_output.atom_array)
    # Convert 3-letter codes to 1-letter using Biotite
seq_1letter = ''.join(
    ProteinSequence.convert_letter_3to1(res_name)
    for res_name in rf3_output.atom_array.res_name[res_starts]
)
print(f"Sequence {i+1}: {seq_1letter}")

Sequence 8: MVINKIITEGSLELSELLKLGASAATIDELIANASVVALNLDSQTLFAATAAGIAYAINTQAGSDLQVCLPKGPTCCSRKMEEKYQLTARLNMEQLLQSASMELKFLIIQNAAVFQEAFEIVVRHAKNYTNAMFKNNYPSLTPQAFEFVGEFFTDVSLYILGSDINVDDMVNELFDSLFPVIYTQLMNPGLPDSALDINECLRGARRDLKVFGNFPKLIMTQVSKSLQVTRIFLQALNLGIEVINTTDHLKFSKDCGRMLTRMWYCSYCQGLMMVKPCGGYCNVVMQGCMAGVVEIDKYWREYILSLEELVNGMYRIYDMENVLLGLFSTIHDSIQYVQKNAGKLTTTIGKLCAHSQQRQYRSAYYPEDLFIDKKVLKVAHVEHEETLSSRRRELIQKLKSFISFYSALPGYICSHSPVAENDTLCWNGQELVERYSQKAARNGMKNQFNLHELKMKGPEPVVSQIIDKLKHINQLLRTMS


In [ ]:
res_starts = get_residue_starts(atom_array)
    # Convert 3-letter codes to 1-letter using Biotite
seq_1letter = ''.join(
    ProteinSequence.convert_letter_3to1(res_name)
    for res_name in atom_array.res_name[res_starts]
)
print(f"Sequence {i+1}: {seq_1letter}")

Sequence 8: MVINKIITEGSLELSELLKLGASAATIDELIANASVVALNLDSQTLFAATAAGIAYAINTQAGSDLQVCLPKGPTCCSRKMEEKYQLTARLNMEQLLQSASMELKFLIIQNAAVFQEAFEIVVRHAKNYTNAMFKNNYPSLTPQAFEFVGEFFTDVSLYILGSDINVDDMVNELFDSLFPVIYTQLMNPGLPDSALDINECLRGARRDLKVFGNFPKLIMTQVSKSLQVTRIFLQALNLGIEVINTTDHLKFSKDCGRMLTRMWYCSYCQGLMMVKPCGGYCNVVMQGCMAGVVEIDKYWREYILSLEELVNGMYRIYDMENVLLGLFSTIHDSIQYVQKNAGKLTTTIGKLCAHSQQRQYRSAYYPEDLFIDKKVLKVAHVEHEETLSSRRRELIQKLKSFISFYSALPGYICSHSPVAENDTLCWNGQELVERYSQKAARNGMKNQFNLHELKMKGPEPVVSQIIDKLKHINQLLRTMS


In [8]:
a = np.random.randint(0, 10, (10, 10))